# **##  Ingest sprints.json **file****

1. Read the file using spark dataframe reader API
2. Add Metadata Columns 
- Source File
- Ingestion Timestamp
3. Write to bronze delta table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
# Define source file and table name:
source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

# **Step 1- read the JSON file using the dataframe reader **API****

In [0]:
# Define Schema :
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType

sprints_schema = StructType([
    StructField('date', DateType()),
    StructField('raceName', StringType()),
    StructField('round', IntegerType()),
    StructField('season', IntegerType()),
    StructField('url', StringType()),
    StructField('constructorId', StringType()),
    StructField('driverId', StringType()),
    StructField('grid', IntegerType()),
    StructField('laps', IntegerType()),
    StructField('number', IntegerType()),
    StructField('points', FloatType()),
    StructField('position', IntegerType()),
    StructField('positionText', StringType()),
    StructField('status', StringType())
])


In [0]:
# Read data:

sprints_df = (
    spark.read
        .format('json')
        # .option('inferSchema','true')
        .option('multiLine', True)
        .schema(sprints_schema)
        .load(source_file)
)

In [0]:
display(sprints_df)

# **Step** **2** - **Add** **Metadata** **Columns**

- Source File 
- Ingestion Timestamp

In [0]:
from pyspark.sql import functions as F

sprints_final_df = add_ingestion_metadata(sprints_df)

In [0]:
display(sprints_final_df)

# **Step 3 - Write to bronze delta table**

In [0]:
(
    sprints_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
%sql
SELECT * FROM formula1.bronze.sprints

In [0]:
display(spark.table(table_name))